In [5]:
# =============================================================================
# BLOCK 1 — LOAD AND AUDIT THE CANONICAL GSE200164 ANNDATA OBJECT
# =============================================================================
#
# Purpose
# -------
# 1. Load the existing .h5ad file without rebuilding the sparse matrix.
# 2. Confirm the number of cells and genes.
# 3. Confirm that the expression matrix is sparse.
# 4. Check whether adata.X contains raw integer-like counts.
# 5. Inspect the available cell metadata.
# 6. Confirm that we have a sample-level column for the 59 donor samples.
#
# This block intentionally makes NO changes to the data.
# =============================================================================


# -----------------------------------------------------------------------------
# 1. Import required packages
# -----------------------------------------------------------------------------

from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp


# -----------------------------------------------------------------------------
# 2. Define the path to the canonical AnnData file
# -----------------------------------------------------------------------------

H5AD_PATH = Path(
    "/home/amithsiri/projects/"
    "Summer-Project-2026/"
    "Summer-Project-2026/"
    "Age_trajectory/"
    "data/"
    "processed/"
    "GSE200164_canonical_input.h5ad"
)


# -----------------------------------------------------------------------------
# 3. Confirm that the file exists before attempting to load it
# -----------------------------------------------------------------------------

if not H5AD_PATH.exists():
    raise FileNotFoundError(
        "The AnnData file was not found at:\n"
        f"{H5AD_PATH}\n\n"
        "Check the path and capitalization."
    )

file_size_gb = H5AD_PATH.stat().st_size / (1024**3)

print("AnnData file found.")
print(f"Path: {H5AD_PATH}")
print(f"File size: {file_size_gb:.3f} GB")


# -----------------------------------------------------------------------------
# 4. Load the complete AnnData object into memory
#
# We are not using backed='r' here because later clustering and normalization
# steps will need writable access to the AnnData object.
#
# On the 64 GB VM, loading this 70,391-cell sparse object should be reasonable.
# -----------------------------------------------------------------------------

print("\nLoading AnnData object...")

adata = ad.read_h5ad(H5AD_PATH)

print("AnnData object loaded successfully.")


# -----------------------------------------------------------------------------
# 5. Display the overall structure
#
# AnnData convention:
#
#   adata.n_obs = number of observations, here individual cells
#   adata.n_vars = number of variables, here genes
#
# The expected dimensions are approximately:
#
#   70,391 cells × 19,689 genes
# -----------------------------------------------------------------------------

print("\nAnnData summary:")
print(adata)

print("\nDimensions:")
print(f"Cells: {adata.n_obs:,}")
print(f"Genes: {adata.n_vars:,}")


# -----------------------------------------------------------------------------
# 6. Check whether the expression matrix is sparse
#
# The full dense matrix would contain:
#
#   70,391 × 19,689 ≈ 1.39 billion values
#
# It must remain sparse. Never run:
#
#   adata.X.toarray()
#   np.asarray(adata.X)
#
# on the complete dataset.
# -----------------------------------------------------------------------------

is_sparse = sp.issparse(adata.X)

print("\nExpression-matrix storage:")
print(f"adata.X type: {type(adata.X)}")
print(f"adata.X is sparse: {is_sparse}")

if not is_sparse:
    estimated_dense_gb = (
        adata.n_obs
        * adata.n_vars
        * np.dtype(adata.X.dtype).itemsize
        / (1024**3)
    )

    print(
        "\nWARNING: adata.X is dense.\n"
        f"Estimated matrix memory: {estimated_dense_gb:.2f} GB"
    )


# -----------------------------------------------------------------------------
# 7. Convert the sparse matrix to CSR format if needed
#
# CSR is efficient for cell-wise operations because rows represent cells.
# This does not densify the matrix.
# -----------------------------------------------------------------------------

if is_sparse and not sp.isspmatrix_csr(adata.X):
    print("\nConverting adata.X to CSR sparse format.")
    adata.X = adata.X.tocsr()

print(f"Final adata.X format: {type(adata.X)}")


# -----------------------------------------------------------------------------
# 8. Calculate the actual memory used by the sparse matrix
#
# A CSR matrix primarily consists of:
#
#   data    — stored nonzero expression values
#   indices — gene index for each stored value
#   indptr  — boundaries marking the beginning of each cell row
# -----------------------------------------------------------------------------

if sp.isspmatrix_csr(adata.X):
    sparse_bytes = (
        adata.X.data.nbytes
        + adata.X.indices.nbytes
        + adata.X.indptr.nbytes
    )

    print("\nSparse-matrix memory:")
    print(f"Nonzero values: {adata.X.nnz:,}")
    print(f"Stored matrix size: {sparse_bytes / 1024**3:.3f} GB")
    print(f"Value dtype: {adata.X.data.dtype}")
    print(f"Index dtype: {adata.X.indices.dtype}")


# -----------------------------------------------------------------------------
# 9. Check cell and gene identifier integrity
#
# Duplicate cell barcodes or gene names can cause incorrect joins, plotting,
# and downstream annotation.
# -----------------------------------------------------------------------------

print("\nIdentifier checks:")
print(f"Cell names unique: {adata.obs_names.is_unique}")
print(f"Gene names unique: {adata.var_names.is_unique}")

if not adata.obs_names.is_unique:
    duplicated_cells = adata.obs_names[
        adata.obs_names.duplicated()
    ]

    print("Example duplicated cell names:")
    print(duplicated_cells[:10].tolist())

if not adata.var_names.is_unique:
    duplicated_genes = adata.var_names[
        adata.var_names.duplicated()
    ]

    print("Example duplicated gene names:")
    print(duplicated_genes[:10].tolist())


# -----------------------------------------------------------------------------
# 10. Inspect metadata columns
#
# We specifically need a column representing the biological sample or donor.
# The authors split the Seurat object using orig.ident before SCTransform and
# integration. Their analysis included 59 subjects/samples.
# -----------------------------------------------------------------------------

print("\nCell metadata columns:")
for column in adata.obs.columns:
    print(f"  - {column}")


# Display a small metadata preview.
display(adata.obs.head())


# -----------------------------------------------------------------------------
# 11. Inspect likely sample-identity columns
#
# Candidate names from the existing data-engineering work include:
#
#   sample_id
#   orig.ident
#   ID
#   gsm
#
# We will not select one automatically yet. We first inspect the number of
# unique values in each candidate.
# -----------------------------------------------------------------------------

candidate_sample_columns = [
    "orig.ident",
    "sample_id",
    "ID",
    "sample",
    "gsm",
]

available_sample_columns = [
    column
    for column in candidate_sample_columns
    if column in adata.obs.columns
]

print("\nCandidate sample columns found:")
print(available_sample_columns)

for column in available_sample_columns:
    unique_count = adata.obs[column].nunique(dropna=False)

    print(
        f"\nColumn: {column}\n"
        f"Unique values: {unique_count:,}"
    )

    print(
        adata.obs[column]
        .value_counts(dropna=False)
        .head(15)
    )


# -----------------------------------------------------------------------------
# 12. Inspect likely clinical metadata columns
# -----------------------------------------------------------------------------

clinical_columns = [
    "diagnosis",
    "Diagnosis",
    "age",
    "sex",
    "gender",
]

for column in clinical_columns:
    if column in adata.obs.columns:
        print(f"\nMetadata summary: {column}")

        print(
            adata.obs[column]
            .value_counts(dropna=False)
            .sort_index()
            .head(30)
        )


# -----------------------------------------------------------------------------
# 13. Audit the expression values
#
# Raw UMI counts are expected to be:
#
#   - nonnegative
#   - predominantly or entirely integer-valued
#
# We examine only the stored nonzero values. This does not densify the matrix.
# -----------------------------------------------------------------------------

if sp.issparse(adata.X):
    stored_values = adata.X.data
else:
    # This branch is included for completeness, but we do not expect a dense
    # matrix for this dataset.
    stored_values = adata.X.ravel()


minimum_value = float(stored_values.min())
maximum_value = float(stored_values.max())

negative_count = int(np.count_nonzero(stored_values < 0))

integer_like = np.isclose(
    stored_values,
    np.round(stored_values),
    rtol=0,
    atol=1e-8,
)

fraction_integer_like = float(integer_like.mean())
fraction_fractional = 1.0 - fraction_integer_like

print("\nExpression-value audit:")
print(f"Minimum stored value: {minimum_value}")
print(f"Maximum stored value: {maximum_value}")
print(f"Negative stored values: {negative_count:,}")
print(f"Integer-like fraction: {fraction_integer_like:.6f}")
print(f"Fractional-value fraction: {fraction_fractional:.6f}")


# -----------------------------------------------------------------------------
# 14. Calculate basic per-cell count statistics
#
# These quantities should match raw-count behavior:
#
#   total_counts       = total UMI count for each cell
#   detected_genes     = number of genes with nonzero expression
#
# These calculations preserve sparsity.
# -----------------------------------------------------------------------------

if sp.isspmatrix_csr(adata.X):
    total_counts = np.asarray(
        adata.X.sum(axis=1)
    ).ravel()

    detected_genes = np.diff(
        adata.X.indptr
    )

else:
    total_counts = np.asarray(
        adata.X.sum(axis=1)
    ).ravel()

    detected_genes = np.count_nonzero(
        adata.X,
        axis=1,
    )


print("\nPer-cell total-count summary:")
print(
    pd.Series(total_counts)
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)


print("\nDetected-gene summary:")
print(
    pd.Series(detected_genes)
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)


# -----------------------------------------------------------------------------
# 15. Check mitochondrial-gene availability
#
# The original paper calculated mitochondrial percentage using genes beginning
# with "MT-". We only check their presence here; we do not filter any cells.
# -----------------------------------------------------------------------------

gene_names = adata.var_names.astype(str)

mt_mask = gene_names.str.startswith("MT-")
n_mt_genes = int(mt_mask.sum())

print("\nMitochondrial-gene check:")
print(f"Genes beginning with 'MT-': {n_mt_genes}")

if n_mt_genes > 0:
    mt_counts = np.asarray(
        adata.X[:, mt_mask].sum(axis=1)
    ).ravel()

    percent_mt = np.divide(
        mt_counts,
        total_counts,
        out=np.zeros_like(mt_counts, dtype=np.float64),
        where=total_counts > 0,
    ) * 100

    print("\nCalculated mitochondrial-percentage summary:")
    print(
        pd.Series(percent_mt)
        .describe(
            percentiles=[
                0.50,
                0.90,
                0.95,
                0.99,
            ]
        )
    )


# -----------------------------------------------------------------------------
# 16. Inspect existing AnnData layers and embeddings
#
# We want to know whether raw counts, normalized values, PCA, or prior UMAP
# coordinates have already been stored elsewhere in the object.
# -----------------------------------------------------------------------------

print("\nExisting AnnData components:")
print(f"Layers: {list(adata.layers.keys())}")
print(f"obsm entries: {list(adata.obsm.keys())}")
print(f"varm entries: {list(adata.varm.keys())}")
print(f"obsp entries: {list(adata.obsp.keys())}")
print(f"uns entries: {list(adata.uns.keys())}")
print(f"adata.raw exists: {adata.raw is not None}")


# -----------------------------------------------------------------------------
# 17. Final validation report
# -----------------------------------------------------------------------------

expected_cells = 70_391
expected_genes = 19_689

print("\n" + "=" * 80)
print("BLOCK 1 VALIDATION REPORT")
print("=" * 80)

print(
    f"Expected dimensions: "
    f"{expected_cells:,} cells × {expected_genes:,} genes"
)

print(
    f"Observed dimensions: "
    f"{adata.n_obs:,} cells × {adata.n_vars:,} genes"
)

print(f"Cell count matches expected: {adata.n_obs == expected_cells}")
print(f"Gene count matches expected: {adata.n_vars == expected_genes}")
print(f"Sparse matrix: {sp.issparse(adata.X)}")
print(f"Unique cell names: {adata.obs_names.is_unique}")
print(f"Unique gene names: {adata.var_names.is_unique}")
print(f"Negative expression values: {negative_count:,}")
print(f"Integer-like expression fraction: {fraction_integer_like:.6f}")
print(f"Candidate sample columns: {available_sample_columns}")

print("=" * 80)
print("BLOCK 1 COMPLETE")
print("=" * 80)

AnnData file found.
Path: /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/data/processed/GSE200164_canonical_input.h5ad
File size: 0.602 GB

Loading AnnData object...
AnnData object loaded successfully.

AnnData summary:
AnnData object with n_obs × n_vars = 70391 × 19689
    obs: 'sample_prefix', 'tenx_barcode', 'patient_id', 'diagnosis', 'sex', 'cell_type', 'age', 'gsm', 'title', 'diagnosis_original', 'cognitive_group'
    var: 'gene_symbol', 'chromosome', 'start', 'end', 'width', 'strand'
    uns: 'clinical_variable_definitions', 'dataset', 'processing'
    layers: None (.X)

Dimensions:
Cells: 70,391
Genes: 19,689

Expression-matrix storage:
adata.X type: <class 'scipy.sparse._csr.csr_matrix'>
adata.X is sparse: True
Final adata.X format: <class 'scipy.sparse._csr.csr_matrix'>

Sparse-matrix memory:
Nonzero values: 126,603,213
Stored matrix size: 0.944 GB
Value dtype: float32
Index dtype: int32

Identifier checks:
Cell names unique: True
Gene names un

,sample_prefix,tenx_barcode,patient_id,diagnosis,sex,cell_type,age,gsm,title,diagnosis_original,cognitive_group
cell_barcode,,,,,,,,,,,
A1_AAACCTGGTTGGTGGA-1,A1,AAACCTGGTTGGTGGA-1,A1,Healthy,Female,unknown,67.0,GSM6016346,CSF_Healthy_A1,Healthy,Healthy
A1_AAACGGGAGAGACGAA-1,A1,AAACGGGAGAGACGAA-1,A1,Healthy,Female,unknown,67.0,GSM6016346,CSF_Healthy_A1,Healthy,Healthy
A1_AAAGCAAAGACTTGAA-1,A1,AAAGCAAAGACTTGAA-1,A1,Healthy,Female,unknown,67.0,GSM6016346,CSF_Healthy_A1,Healthy,Healthy
A1_AAAGCAAAGGCAGTCA-1,A1,AAAGCAAAGGCAGTCA-1,A1,Healthy,Female,unknown,67.0,GSM6016346,CSF_Healthy_A1,Healthy,Healthy
A1_AACACGTAGCCAGAAC-1,A1,AACACGTAGCCAGAAC-1,A1,Healthy,Female,unknown,67.0,GSM6016346,CSF_Healthy_A1,Healthy,Healthy



Candidate sample columns found:
['gsm']

Column: gsm
Unique values: 59
gsm
GSM6016383    4645
GSM6016347    4169
GSM6016401    3672
GSM6016390    2542
GSM6016377    2467
GSM6016399    2421
GSM6016361    2362
GSM6016397    2249
GSM6016375    2219
GSM6016369    2206
GSM6016385    1925
GSM6016348    1682
GSM6016382    1601
GSM6016359    1569
GSM6016368    1453
Name: count, dtype: int64

Metadata summary: diagnosis
diagnosis
Healthy    54267
MCI/AD     16124
Name: count, dtype: int64

Metadata summary: age
age
48.0     331
54.0    1864
55.0     497
56.0    1398
62.0    1250
63.0    5456
64.0    5309
65.0    3920
66.0    2952
67.0    5570
69.0    4796
70.0    3986
71.0    1187
72.0    5527
73.0    6094
74.0    2813
75.0    3672
76.0    2043
77.0    4476
78.0    1924
79.0    3090
80.0     635
81.0     611
82.0     990
Name: count, dtype: int64

Metadata summary: sex
sex
Female    39299
Male      31092
Name: count, dtype: int64

Expression-value audit:
Minimum stored value: 3.754570207092911

In [6]:
# =============================================================================
# CELL 2A — SELECT THE 59-SAMPLE GROUPING COLUMN
# =============================================================================

SAMPLE_KEY = "gsm"

if SAMPLE_KEY not in adata.obs.columns:
    raise KeyError(
        f"{SAMPLE_KEY!r} is not present in adata.obs."
    )

n_samples = adata.obs[SAMPLE_KEY].nunique(dropna=False)
n_missing = adata.obs[SAMPLE_KEY].isna().sum()

print(f"Selected sample key: {SAMPLE_KEY}")
print(f"Unique sample values: {n_samples}")
print(f"Missing values: {n_missing:,}")

if n_samples != 59:
    raise ValueError(
        f"Expected 59 samples, but found {n_samples}."
    )

if n_missing != 0:
    raise ValueError(
        f"Found {n_missing} cells with missing sample identity."
    )

adata.obs[SAMPLE_KEY] = adata.obs[SAMPLE_KEY].astype("category")

sample_cell_counts = (
    adata.obs[SAMPLE_KEY]
    .value_counts()
    .sort_index()
)

print("\nCells per sample:")
display(sample_cell_counts.to_frame(name="n_cells"))

print("\nSample-count summary:")
display(
    sample_cell_counts
    .describe()
    .to_frame(name="cells_per_sample")
)

Selected sample key: gsm
Unique sample values: 59
Missing values: 0

Cells per sample:


,n_cells
gsm,
GSM6016346,684
GSM6016347,4169
GSM6016348,1682
GSM6016349,1001
GSM6016350,1316
GSM6016351,1239
GSM6016352,151
GSM6016353,966
GSM6016354,843



Sample-count summary:


,cells_per_sample
count,59.000000
mean,1193.067797
std,946.411131
min,151.000000
25%,569.000000
50%,966.000000
75%,1441.500000
max,4645.000000


In [7]:
# =============================================================================
# CELL 2 — DEFINE SAMPLE IDENTITY, PRESERVE COUNTS, AND STORE QC METRICS
# =============================================================================
#
# Purpose
# -------
# 1. Identify the metadata column corresponding to the authors' "orig.ident".
# 2. Verify that it represents the 59 biological samples.
# 3. Preserve the current SoupX-corrected count matrix in a dedicated layer.
# 4. Calculate and store the QC variables used by the authors:
#
#       total counts per cell
#       detected genes per cell
#       mitochondrial percentage
#
# 5. Confirm that the AnnData object appears to contain the already-filtered
#    70,391-cell dataset.
#
# This cell does NOT normalize, integrate, run PCA, cluster, or run UMAP.
# =============================================================================




# -----------------------------------------------------------------------------
# 4. Preserve the current expression matrix as the count layer
#
# adata.X will eventually be transformed for normalization and dimensionality
# reduction. We therefore preserve the current processed counts separately.
#
# Important:
# These may be fractional because SoupX subtracts estimated ambient RNA.
# Fractional SoupX-adjusted counts are still count-like input values.
# -----------------------------------------------------------------------------

if "counts" in adata.layers:
    print(
        "\nA layer named 'counts' already exists."
        "\nIt will not be overwritten."
    )

    counts_layer = adata.layers["counts"]

    if counts_layer.shape != adata.X.shape:
        raise ValueError(
            "The existing counts layer does not have the same shape as adata.X."
        )

else:
    print("\nCreating adata.layers['counts'] from the current adata.X matrix.")

    # copy() preserves sparsity and ensures the layer does not point to the same
    # mutable matrix object as adata.X.
    adata.layers["counts"] = adata.X.copy()


# Confirm that the layer remains sparse.
print("\nCount-layer audit:")
print(f"Type: {type(adata.layers['counts'])}")
print(f"Shape: {adata.layers['counts'].shape}")
print(f"Sparse: {sp.issparse(adata.layers['counts'])}")


if not sp.issparse(adata.layers["counts"]):
    raise TypeError(
        "adata.layers['counts'] is dense. Stop before continuing."
    )


# Convert to CSR if necessary because cells occupy rows.
if not sp.isspmatrix_csr(adata.layers["counts"]):
    print("Converting the counts layer to CSR format.")
    adata.layers["counts"] = adata.layers["counts"].tocsr()


# -----------------------------------------------------------------------------
# 5. Calculate the authors' QC covariates from the preserved counts
#
# In Seurat, these correspond approximately to:
#
#     nCount_RNA
#     nFeature_RNA
#     percent.mt
#
# The authors regressed these quantities during SCTransform.
# -----------------------------------------------------------------------------

counts = adata.layers["counts"]


# Total SoupX-adjusted counts per cell.
adata.obs["nCount_RNA"] = np.asarray(
    counts.sum(axis=1)
).ravel()


# Number of genes with nonzero expression in each cell.
adata.obs["nFeature_RNA"] = np.diff(
    counts.indptr
)


# Identify mitochondrial genes.
mt_gene_mask = (
    adata.var_names
    .astype(str)
    .str.startswith("MT-")
)


n_mt_genes = int(mt_gene_mask.sum())

print(f"\nNumber of MT- genes: {n_mt_genes}")


if n_mt_genes == 0:
    raise ValueError(
        "No genes beginning with 'MT-' were found.\n"
        "Mitochondrial percentage cannot be reproduced until gene naming "
        "is resolved."
    )


# Sum mitochondrial counts for each cell.
mt_counts = np.asarray(
    counts[:, mt_gene_mask].sum(axis=1)
).ravel()


# Calculate mitochondrial percentage safely.
adata.obs["percent.mt"] = np.divide(
    mt_counts,
    adata.obs["nCount_RNA"].to_numpy(),
    out=np.zeros(
        adata.n_obs,
        dtype=np.float64,
    ),
    where=adata.obs["nCount_RNA"].to_numpy() > 0,
) * 100


# -----------------------------------------------------------------------------
# 6. Summarize the QC variables
# -----------------------------------------------------------------------------

qc_columns = [
    "nCount_RNA",
    "nFeature_RNA",
    "percent.mt",
]

print("\nQC summary:")
display(
    adata.obs[qc_columns].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    ).T
)


# -----------------------------------------------------------------------------
# 7. Check whether the published QC filters appear already satisfied
#
# The authors retained cells with:
#
#     at least 200 detected genes
#     less than 10% mitochondrial reads
#     DoubletFinder classification of "Singlet"
#
# We do not filter here because the published processed matrix is expected to
# contain the final 70,391 quality-controlled cells already.
# -----------------------------------------------------------------------------

below_200_features = int(
    (adata.obs["nFeature_RNA"] < 200).sum()
)

at_or_above_10_percent_mt = int(
    (adata.obs["percent.mt"] >= 10).sum()
)


print("\nPublished-QC threshold audit:")
print(f"Cells with fewer than 200 genes: {below_200_features:,}")
print(
    "Cells with mitochondrial percentage >= 10%: "
    f"{at_or_above_10_percent_mt:,}"
)


if below_200_features > 0 or at_or_above_10_percent_mt > 0:
    print(
        "\nWARNING:"
        "\nSome cells do not satisfy the thresholds described in the paper."
        "\nDo not remove them automatically yet."
        "\nFirst determine whether this difference is caused by:"
        "\n  - how SoupX fractional counts were stored,"
        "\n  - gene-name differences,"
        "\n  - or use of a different GEO matrix."
    )
else:
    print(
        "\nAll cells satisfy the measurable published QC thresholds."
    )


# -----------------------------------------------------------------------------
# 8. Check whether a DoubletFinder classification is available
#
# This may not be present in the GEO metadata, even though doublet removal was
# performed before publication.
# -----------------------------------------------------------------------------

doublet_columns = [
    column
    for column in adata.obs.columns
    if (
        "doublet" in column.lower()
        or "df.class" in column.lower()
        or "singlet" in column.lower()
    )
]


print("\nPossible doublet metadata columns:")
print(doublet_columns)


for column in doublet_columns:
    print(f"\n{column}:")
    print(
        adata.obs[column]
        .value_counts(dropna=False)
        .head(20)
    )


# -----------------------------------------------------------------------------
# 9. Confirm clinical metadata is constant within each sample
#
# Age, sex, and diagnosis should describe the donor/sample, not vary from cell
# to cell within one sample.
# -----------------------------------------------------------------------------

sample_level_columns = [
    column
    for column in [
        "sample_id",
        "ID",
        "diagnosis",
        "Diagnosis",
        "age",
        "sex",
    ]
    if column in adata.obs.columns
]


print("\nSample-level metadata consistency:")


for column in sample_level_columns:

    values_per_sample = (
        adata.obs
        .groupby(SAMPLE_KEY, observed=True)[column]
        .nunique(dropna=False)
    )

    inconsistent_samples = values_per_sample[
        values_per_sample > 1
    ]

    print(
        f"{column}: "
        f"{len(inconsistent_samples)} samples with multiple values"
    )

    if len(inconsistent_samples) > 0:
        display(inconsistent_samples.head(20))


# -----------------------------------------------------------------------------
# 10. Store pipeline information in adata.uns
#
# This documents where the matrix came from and prevents future notebook cells
# from silently treating it as an unprocessed Cell Ranger matrix.
# -----------------------------------------------------------------------------

adata.uns["figure1d_reproduction"] = {
    "dataset": "GSE200164",
    "sample_key": SAMPLE_KEY,
    "expected_samples": 59,
    "expected_cells": 70_391,
    "expected_genes": 19_689,
    "input_layer": "counts",
    "input_interpretation": (
        "Processed, SoupX-adjusted, quality-controlled count-like matrix"
    ),
    "authors_initial_pcs": 15,
    "authors_umap_neighbors": 30,
    "authors_cluster_resolution": 0.3,
    "authors_variable_features_per_sample": 1000,
}


# -----------------------------------------------------------------------------
# 11. Final Cell 2 report
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("CELL 2 VALIDATION REPORT")
print("=" * 80)

print(f"Sample key: {SAMPLE_KEY}")
print(
    "Number of biological samples: "
    f"{adata.obs[SAMPLE_KEY].nunique()}"
)
print(f"Counts layer exists: {'counts' in adata.layers}")
print(
    "Counts layer sparse: "
    f"{sp.issparse(adata.layers['counts'])}"
)
print(
    "Cells with fewer than 200 genes: "
    f"{below_200_features:,}"
)
print(
    "Cells with mitochondrial percentage >=10%: "
    f"{at_or_above_10_percent_mt:,}"
)
print(
    "Observed dimensions: "
    f"{adata.n_obs:,} cells × {adata.n_vars:,} genes"
)

print("=" * 80)
print("CELL 2 COMPLETE")
print("=" * 80)


Creating adata.layers['counts'] from the current adata.X matrix.

Count-layer audit:
Type: <class 'scipy.sparse._csr.csr_matrix'>
Shape: (70391, 19689)
Sparse: True

Number of MT- genes: 13

QC summary:


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
nCount_RNA,70391.0,5159.911621,3130.106201,469.830658,1234.217261,1899.724854,3224.463989,4376.125977,5902.6250,12032.701660,16547.987500,42120.402344
nFeature_RNA,70391.0,1798.571025,697.386006,202.000000,608.000000,871.000000,1344.000000,1691.000000,2071.0000,3267.000000,3977.000000,6687.000000
percent.mt,70391.0,4.638179,1.615651,0.000000,1.240237,2.229734,3.543624,4.485274,5.6033,7.594413,9.079332,9.997854



Published-QC threshold audit:
Cells with fewer than 200 genes: 0
Cells with mitochondrial percentage >= 10%: 0

All cells satisfy the measurable published QC thresholds.

Possible doublet metadata columns:
[]

Sample-level metadata consistency:
diagnosis: 0 samples with multiple values
age: 0 samples with multiple values
sex: 0 samples with multiple values

CELL 2 VALIDATION REPORT
Sample key: gsm
Number of biological samples: 59
Counts layer exists: True
Counts layer sparse: True
Cells with fewer than 200 genes: 0
Cells with mitochondrial percentage >=10%: 0
Observed dimensions: 70,391 cells × 19,689 genes
CELL 2 COMPLETE


In [8]:
# =============================================================================
# CELL 3 — BUILD A SAMPLE-LEVEL METADATA TABLE AND RECOVER INTERNAL SAMPLE IDS
# =============================================================================
#
# Purpose
# -------
# 1. Collapse cell-level metadata into one row per biological sample.
# 2. Inspect the relationship between:
#
#       gsm
#       sample_id
#       orig.ident
#       title
#       barcode prefixes
#
# 3. Determine whether the authors' internal IDs, such as A4 and A6, can be
#    recovered.
# 4. Save a sample-level metadata table for later integration and analysis.
#
# This cell does not modify the expression matrix.
# =============================================================================


from pathlib import Path
import re

import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# 1. Define an output directory for Figure 1D reproduction artifacts
# -----------------------------------------------------------------------------

FIGURE1D_DIR = Path(
    "/home/amithsiri/projects/"
    "Summer-Project-2026/"
    "Summer-Project-2026/"
    "Age_trajectory/"
    "results/"
    "figure1d_python"
)

FIGURE1D_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"Output directory: {FIGURE1D_DIR}")


# -----------------------------------------------------------------------------
# 2. Identify metadata columns that may describe biological samples
# -----------------------------------------------------------------------------

candidate_metadata_columns = [
    "gsm",
    "sample_id",
    "orig.ident",
    "ID",
    "title",
    "diagnosis",
    "Diagnosis",
    "age",
    "sex",
    "gender",
    "n_cells",
]

available_metadata_columns = [
    column
    for column in candidate_metadata_columns
    if column in adata.obs.columns
]

print("\nAvailable sample-related metadata columns:")

for column in available_metadata_columns:
    print(f"  - {column}")


# -----------------------------------------------------------------------------
# 3. Recover a possible internal sample ID from each cell barcode
#
# The authors generated cell names with sample prefixes such as:
#
#     A1_AAAC...
#     A4_AAAC...
#     A6_AAAC...
#
# We test the text before the first underscore.
# -----------------------------------------------------------------------------

cell_names = pd.Index(
    adata.obs_names.astype(str)
)

barcode_prefix = (
    cell_names
    .str.split("_")
    .str[0]
)


# Internal sample IDs in the authors' scripts follow a pattern resembling:
#
#     one capital letter followed by one or more digits
#
# Examples:
#
#     A1
#     A4
#     H8
#
internal_id_pattern = re.compile(
    r"^[A-Z]\d+$"
)

prefix_matches_internal_pattern = np.array(
    [
        bool(internal_id_pattern.match(prefix))
        for prefix in barcode_prefix
    ]
)


print("\nBarcode-prefix audit:")
print(
    "Unique prefixes before the first underscore: "
    f"{pd.Series(barcode_prefix).nunique():,}"
)

print(
    "Cells whose prefix resembles an internal sample ID: "
    f"{prefix_matches_internal_pattern.sum():,} / {adata.n_obs:,}"
)


print("\nFirst 20 barcode prefixes:")
print(
    pd.Series(barcode_prefix)
    .head(20)
    .tolist()
)


print("\nMost common barcode prefixes:")
display(
    pd.Series(
        barcode_prefix,
        name="barcode_prefix",
    )
    .value_counts()
    .head(30)
    .to_frame("n_cells")
)


# -----------------------------------------------------------------------------
# 4. Store the prefix temporarily for sample-level inspection
#
# We only promote this to an official internal ID if it maps cleanly one-to-one
# with the 59 GSM samples.
# -----------------------------------------------------------------------------

adata.obs["barcode_prefix_audit"] = pd.Categorical(
    barcode_prefix
)


# -----------------------------------------------------------------------------
# 5. Verify that candidate metadata fields are constant within GSM
#
# Each GSM should describe one biological sample. Therefore, fields such as age,
# sex, diagnosis, title, and internal ID should have at most one value per GSM.
# -----------------------------------------------------------------------------

fields_to_check = [
    column
    for column in [
        "sample_id",
        "orig.ident",
        "ID",
        "title",
        "diagnosis",
        "Diagnosis",
        "age",
        "sex",
        "gender",
        "barcode_prefix_audit",
    ]
    if column in adata.obs.columns
]


print("\nWithin-GSM consistency checks:")

for column in fields_to_check:

    unique_values_per_gsm = (
        adata.obs
        .groupby(
            "gsm",
            observed=True,
        )[column]
        .nunique(
            dropna=False
        )
    )

    n_inconsistent = int(
        (unique_values_per_gsm > 1).sum()
    )

    print(
        f"{column}: "
        f"{n_inconsistent} GSM samples contain multiple values"
    )


# -----------------------------------------------------------------------------
# 6. Create one row per GSM
#
# Since the previous check established whether each field is constant within a
# GSM, we can safely select the first value for consistent fields.
# -----------------------------------------------------------------------------

sample_table_columns = [
    column
    for column in [
        "gsm",
        "sample_id",
        "orig.ident",
        "ID",
        "title",
        "diagnosis",
        "Diagnosis",
        "age",
        "sex",
        "gender",
        "n_cells",
        "barcode_prefix_audit",
    ]
    if column in adata.obs.columns
]


sample_table = (
    adata.obs[
        sample_table_columns
    ]
    .reset_index(
        names="cell_barcode"
    )
    .groupby(
        "gsm",
        observed=True,
        as_index=False,
    )
    .first()
)


# Calculate the observed cell count directly from the AnnData object.
observed_cell_counts = (
    adata.obs["gsm"]
    .value_counts()
    .rename("observed_cells")
)


sample_table = (
    sample_table
    .merge(
        observed_cell_counts,
        left_on="gsm",
        right_index=True,
        how="left",
        validate="one_to_one",
    )
)


# Sort by a possible internal ID when available.
if "barcode_prefix_audit" in sample_table.columns:
    sample_table = sample_table.sort_values(
        [
            "barcode_prefix_audit",
            "gsm",
        ]
    )
else:
    sample_table = sample_table.sort_values(
        "gsm"
    )


sample_table = sample_table.reset_index(
    drop=True
)


print("\nSample-level metadata table:")
display(sample_table)


# -----------------------------------------------------------------------------
# 7. Test whether the barcode prefix maps one-to-one with GSM
# -----------------------------------------------------------------------------

prefix_mapping_is_valid = False


if "barcode_prefix_audit" in sample_table.columns:

    n_unique_prefixes = (
        sample_table["barcode_prefix_audit"]
        .astype(str)
        .nunique()
    )

    n_unique_gsms = (
        sample_table["gsm"]
        .astype(str)
        .nunique()
    )

    prefix_pattern_fraction = (
        sample_table["barcode_prefix_audit"]
        .astype(str)
        .str.match(
            r"^[A-Z]\d+$"
        )
        .mean()
    )

    print("\nGSM-to-prefix mapping audit:")
    print(f"Unique GSM values: {n_unique_gsms}")
    print(f"Unique barcode prefixes: {n_unique_prefixes}")
    print(
        "Fraction of prefixes matching the authors' ID pattern: "
        f"{prefix_pattern_fraction:.3f}"
    )

    prefix_mapping_is_valid = (
        n_unique_gsms == 59
        and n_unique_prefixes == 59
        and prefix_pattern_fraction == 1.0
    )


# -----------------------------------------------------------------------------
# 8. Promote the barcode prefix to internal_sample_id only if valid
# -----------------------------------------------------------------------------

if prefix_mapping_is_valid:

    adata.obs["internal_sample_id"] = pd.Categorical(
        adata.obs["barcode_prefix_audit"].astype(str)
    )

    sample_table["internal_sample_id"] = (
        sample_table["barcode_prefix_audit"]
        .astype(str)
    )

    print(
        "\nSuccessfully recovered a one-to-one mapping between "
        "the 59 GSM samples and the authors' internal sample IDs."
    )

    print("\nIntegration-reference search:")

    for reference_id in [
        "A4",
        "A6",
    ]:

        matching_rows = sample_table[
            sample_table["internal_sample_id"]
            == reference_id
        ]

        print(f"\nReference sample {reference_id}:")

        if len(matching_rows) == 1:
            display(matching_rows)
        else:
            print(
                "Not found uniquely in the reconstructed mapping."
            )

else:

    print(
        "\nThe barcode prefix cannot yet be treated as the authors' "
        "internal sample ID."
    )

    print(
        "We can still integrate by GSM, but exact reconstruction of the "
        "A4/A6 reference choice will require another mapping source."
    )


# -----------------------------------------------------------------------------
# 9. Save the sample-level table
# -----------------------------------------------------------------------------

SAMPLE_TABLE_PATH = (
    FIGURE1D_DIR
    / "GSE200164_sample_metadata_mapping.csv"
)


sample_table.to_csv(
    SAMPLE_TABLE_PATH,
    index=False,
)


print(
    "\nSaved sample metadata table to:\n"
    f"{SAMPLE_TABLE_PATH}"
)


# -----------------------------------------------------------------------------
# 10. Save a lightweight AnnData checkpoint
#
# The expression matrix remains sparse. This checkpoint preserves:
#
#     counts layer
#     calculated QC values
#     sample grouping
#     recovered internal ID, if available
# -----------------------------------------------------------------------------

CHECKPOINT_PATH = (
    FIGURE1D_DIR
    / "GSE200164_after_qc_audit.h5ad"
)


print("\nWriting checkpoint...")

adata.write_h5ad(
    CHECKPOINT_PATH,
    compression="gzip",
)


print(
    "Checkpoint saved to:\n"
    f"{CHECKPOINT_PATH}"
)


# -----------------------------------------------------------------------------
# 11. Final report
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("CELL 3 VALIDATION REPORT")
print("=" * 80)

print(f"Sample table rows: {len(sample_table)}")
print(
    "One-to-one internal-ID mapping recovered: "
    f"{prefix_mapping_is_valid}"
)

if prefix_mapping_is_valid:
    print(
        "Internal sample IDs: "
        f"{sample_table['internal_sample_id'].nunique()}"
    )

    print(
        "A4 found: "
        f"{(sample_table['internal_sample_id'] == 'A4').sum() == 1}"
    )

    print(
        "A6 found: "
        f"{(sample_table['internal_sample_id'] == 'A6').sum() == 1}"
    )

print(f"Sample table saved: {SAMPLE_TABLE_PATH}")
print(f"AnnData checkpoint saved: {CHECKPOINT_PATH}")

print("=" * 80)
print("CELL 3 COMPLETE")
print("=" * 80)

Output directory: /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/results/figure1d_python

Available sample-related metadata columns:
  - gsm
  - title
  - diagnosis
  - age
  - sex

Barcode-prefix audit:
Unique prefixes before the first underscore: 59
Cells whose prefix resembles an internal sample ID: 70,391 / 70,391

First 20 barcode prefixes:
['A1', 'A1', 'A1', 'A1', 'A1', 'A1', 'A1', 'A1', 'A1', 'A1', 'A1', 'A1', 'A1', 'A1', 'A1', 'A1', 'A1', 'A1', 'A1', 'A1']

Most common barcode prefixes:


,n_cells
barcode_prefix,
F2,4645
A2,4169
H5,3672
G2,2542
E3,2467
H3,2421
C1,2362
H1,2249
E1,2219



Within-GSM consistency checks:
title: 0 GSM samples contain multiple values
diagnosis: 0 GSM samples contain multiple values
age: 0 GSM samples contain multiple values
sex: 0 GSM samples contain multiple values
barcode_prefix_audit: 0 GSM samples contain multiple values

Sample-level metadata table:


,gsm,cell_barcode,title,diagnosis,age,sex,barcode_prefix_audit,observed_cells
0,GSM6016346,A1_AAACCTGGTTGGTGGA-1,CSF_Healthy_A1,Healthy,67.0,Female,A1,684
1,GSM6016347,A2_AAACCTGAGCAGACTG-1,CSF_Healthy_A2,Healthy,77.0,Male,A2,4169
2,GSM6016348,A4_AAACCTGAGCCAACAG-1,CSF_Healthy_A4,Healthy,67.0,Male,A4,1682
3,GSM6016349,A5_AAACCTGTCAACACAC-1,CSF_Healthy_A5,Healthy,72.0,Male,A5,1001
4,GSM6016350,A6_AAACCTGCACCTATCC-1,CSF_MCI/AD_A6,MCI/AD,78.0,Male,A6,1316
5,GSM6016351,A7_AAACCTGCACCACGTG-1,CSF_Healthy_A7,Healthy,63.0,Female,A7,1239
6,GSM6016352,A8_AAAGATGCAGTTCCCT-1,CSF_Healthy_A8,Healthy,71.0,Female,A8,151
7,GSM6016353,B1_AAACCTGCACATCTTT-1,CSF_MCI/AD_B1,MCI/AD,70.0,Male,B1,966
8,GSM6016354,B2_AAACCTGAGTTAACGA-1,CSF_Healthy_B2,Healthy,67.0,Female,B2,843
9,GSM6016355,B3_AAACCTGGTCCGAGTC-1,CSF_Healthy_B3,Healthy,71.0,Male,B3,668



GSM-to-prefix mapping audit:
Unique GSM values: 59
Unique barcode prefixes: 59
Fraction of prefixes matching the authors' ID pattern: 1.000

Successfully recovered a one-to-one mapping between the 59 GSM samples and the authors' internal sample IDs.

Integration-reference search:

Reference sample A4:


,gsm,cell_barcode,title,diagnosis,age,sex,barcode_prefix_audit,observed_cells,internal_sample_id
2,GSM6016348,A4_AAACCTGAGCCAACAG-1,CSF_Healthy_A4,Healthy,67.0,Male,A4,1682,A4



Reference sample A6:


,gsm,cell_barcode,title,diagnosis,age,sex,barcode_prefix_audit,observed_cells,internal_sample_id
4,GSM6016350,A6_AAACCTGCACCTATCC-1,CSF_MCI/AD_A6,MCI/AD,78.0,Male,A6,1316,A6



Saved sample metadata table to:
/home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/results/figure1d_python/GSE200164_sample_metadata_mapping.csv

Writing checkpoint...
Checkpoint saved to:
/home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/results/figure1d_python/GSE200164_after_qc_audit.h5ad

CELL 3 VALIDATION REPORT
Sample table rows: 59
One-to-one internal-ID mapping recovered: True
Internal sample IDs: 59
A4 found: True
A6 found: True
Sample table saved: /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/results/figure1d_python/GSE200164_sample_metadata_mapping.csv
AnnData checkpoint saved: /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/results/figure1d_python/GSE200164_after_qc_audit.h5ad
CELL 3 COMPLETE


In [9]:
# =============================================================================
# CELL 4 — VERIFY THE R/SEURAT ENVIRONMENT FOR FIGURE 1D REPRODUCTION
# =============================================================================
#
# Purpose
# -------
# The authors generated the Figure 1D clusters with Seurat rather than Scanpy.
#
# Before starting the expensive computation, this cell verifies that:
#
#   1. R is installed and callable from the Python notebook.
#   2. Seurat is installed.
#   3. Supporting R packages are installed.
#   4. zellkonverter is available to read the existing .h5ad file.
#   5. glmGamPoi is available for memory-efficient SCTransform.
#   6. The VM has enough currently available RAM and disk space.
#
# This cell does not modify adata or run normalization.
# =============================================================================


# -----------------------------------------------------------------------------
# 1. Import Python packages used for system inspection
# -----------------------------------------------------------------------------

import json
import shutil
import subprocess
import sys
from pathlib import Path

import psutil


# -----------------------------------------------------------------------------
# 2. Locate the R executable
# -----------------------------------------------------------------------------

rscript_path = shutil.which("Rscript")
r_path = shutil.which("R")


print("=" * 80)
print("R EXECUTABLE CHECK")
print("=" * 80)

print(f"R path:       {r_path}")
print(f"Rscript path: {rscript_path}")


if rscript_path is None:
    raise RuntimeError(
        "Rscript was not found in the current environment.\n"
        "The exact Seurat workflow cannot run until R is available."
    )


# -----------------------------------------------------------------------------
# 3. Print the installed R version
# -----------------------------------------------------------------------------

r_version_result = subprocess.run(
    [
        rscript_path,
        "--version",
    ],
    capture_output=True,
    text=True,
)


# Depending on the R installation, version information may be written to either
# standard output or standard error.
r_version_text = (
    r_version_result.stdout.strip()
    or r_version_result.stderr.strip()
)


print("\nInstalled R version:")
print(r_version_text)


# -----------------------------------------------------------------------------
# 4. Define the required R packages
#
# Core packages
# -------------
# Seurat:
#     Reproduces the authors' SCTransform, integration, PCA, clustering, and UMAP.
#
# Matrix:
#     Sparse-matrix support.
#
# Data-transfer packages
# ----------------------
# zellkonverter:
#     Reads the existing AnnData .h5ad file into R.
#
# SingleCellExperiment:
#     Intermediate Bioconductor representation used by zellkonverter.
#
# glmGamPoi:
#     Faster and more memory-efficient negative-binomial fitting for
#     SCTransform.
#
# Optional package
# ----------------
# future:
#     Controls parallel execution and memory limits used by Seurat.
# -----------------------------------------------------------------------------

required_r_packages = [
    "Seurat",
    "Matrix",
    "zellkonverter",
    "SingleCellExperiment",
    "glmGamPoi",
    "future",
]


# -----------------------------------------------------------------------------
# 5. Ask R which packages are installed and report their versions
# -----------------------------------------------------------------------------

r_package_check_code = r'''
required_packages <- c(
    "Seurat",
    "Matrix",
    "zellkonverter",
    "SingleCellExperiment",
    "glmGamPoi",
    "future"
)

results <- lapply(required_packages, function(pkg) {

    installed <- requireNamespace(
        pkg,
        quietly = TRUE
    )

    version <- if (installed) {
        as.character(
            packageVersion(pkg)
        )
    } else {
        NA_character_
    }

    data.frame(
        package = pkg,
        installed = installed,
        version = version,
        stringsAsFactors = FALSE
    )
})

results <- do.call(
    rbind,
    results
)

write.table(
    results,
    file = stdout(),
    sep = "\t",
    row.names = FALSE,
    quote = FALSE
)
'''


package_check_result = subprocess.run(
    [
        rscript_path,
        "-e",
        r_package_check_code,
    ],
    capture_output=True,
    text=True,
)


if package_check_result.returncode != 0:
    print("\nR package-check error output:")
    print(package_check_result.stderr)

    raise RuntimeError(
        "R was found, but the package audit failed."
    )


print("\n" + "=" * 80)
print("R PACKAGE CHECK")
print("=" * 80)

print(package_check_result.stdout)


# -----------------------------------------------------------------------------
# 6. Parse the package-check result in Python
# -----------------------------------------------------------------------------

package_rows = []


for line in package_check_result.stdout.strip().splitlines():

    # Skip the table header.
    if line.startswith("package\t"):
        continue

    fields = line.split("\t")

    if len(fields) != 3:
        continue

    package_name, installed_text, version = fields

    package_rows.append(
        {
            "package": package_name,
            "installed": installed_text == "TRUE",
            "version": None if version == "NA" else version,
        }
    )


package_status = {
    row["package"]: row
    for row in package_rows
}


missing_r_packages = [
    package
    for package in required_r_packages
    if (
        package not in package_status
        or not package_status[package]["installed"]
    )
]


print("Parsed package status:")

for package in required_r_packages:

    status = package_status.get(
        package,
        {
            "installed": False,
            "version": None,
        },
    )

    print(
        f"  {package:<24} "
        f"installed={status['installed']!s:<5} "
        f"version={status['version']}"
    )


# -----------------------------------------------------------------------------
# 7. Inspect available system memory
# -----------------------------------------------------------------------------

memory = psutil.virtual_memory()

total_ram_gb = memory.total / (1024**3)
available_ram_gb = memory.available / (1024**3)
used_ram_gb = memory.used / (1024**3)


print("\n" + "=" * 80)
print("MEMORY CHECK")
print("=" * 80)

print(f"Total RAM:     {total_ram_gb:.2f} GB")
print(f"Used RAM:      {used_ram_gb:.2f} GB")
print(f"Available RAM: {available_ram_gb:.2f} GB")
print(f"Memory usage:  {memory.percent:.1f}%")


# The full SCT integration is likely unsafe if the notebook begins with very
# little memory available.
if available_ram_gb < 40:
    print(
        "\nWARNING:"
        "\nFewer than 40 GB of RAM are currently available."
        "\nClose other notebooks or processes before running full integration."
    )


# -----------------------------------------------------------------------------
# 8. Inspect available disk space
# -----------------------------------------------------------------------------

disk_usage = shutil.disk_usage(
    FIGURE1D_DIR
)

disk_total_gb = disk_usage.total / (1024**3)
disk_used_gb = disk_usage.used / (1024**3)
disk_free_gb = disk_usage.free / (1024**3)


print("\n" + "=" * 80)
print("DISK CHECK")
print("=" * 80)

print(f"Disk total: {disk_total_gb:.2f} GB")
print(f"Disk used:  {disk_used_gb:.2f} GB")
print(f"Disk free:  {disk_free_gb:.2f} GB")


if disk_free_gb < 15:
    print(
        "\nWARNING:"
        "\nFewer than 15 GB of disk space are available."
        "\nThe Seurat checkpoints and exported embeddings may not fit."
    )


# -----------------------------------------------------------------------------
# 9. Confirm that the AnnData checkpoint exists
# -----------------------------------------------------------------------------

if "CHECKPOINT_PATH" not in globals():
    CHECKPOINT_PATH = (
        FIGURE1D_DIR
        / "GSE200164_after_qc_audit.h5ad"
    )


print("\n" + "=" * 80)
print("INPUT CHECKPOINT")
print("=" * 80)

print(f"Checkpoint path: {CHECKPOINT_PATH}")
print(f"Checkpoint exists: {CHECKPOINT_PATH.exists()}")


if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "The Cell 3 AnnData checkpoint was not found."
    )


checkpoint_size_gb = (
    CHECKPOINT_PATH.stat().st_size
    / (1024**3)
)

print(
    f"Checkpoint size: {checkpoint_size_gb:.3f} GB"
)


# -----------------------------------------------------------------------------
# 10. Define paths that the R integration will use
# -----------------------------------------------------------------------------

R_SCRIPT_PATH = (
    FIGURE1D_DIR
    / "run_figure1d_seurat_integration.R"
)

SEURAT_CHECKPOINT_PATH = (
    FIGURE1D_DIR
    / "GSE200164_seurat_integrated.rds"
)

SEURAT_OUTPUT_DIR = (
    FIGURE1D_DIR
    / "seurat_outputs"
)

SEURAT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print("\nPlanned R outputs:")
print(f"R script:          {R_SCRIPT_PATH}")
print(f"Seurat checkpoint: {SEURAT_CHECKPOINT_PATH}")
print(f"Output directory:  {SEURAT_OUTPUT_DIR}")


# -----------------------------------------------------------------------------
# 11. Save the environment audit
# -----------------------------------------------------------------------------

environment_audit = {
    "python_version": sys.version,
    "rscript_path": rscript_path,
    "r_version": r_version_text,
    "required_r_packages": package_status,
    "missing_r_packages": missing_r_packages,
    "total_ram_gb": total_ram_gb,
    "available_ram_gb": available_ram_gb,
    "disk_free_gb": disk_free_gb,
    "input_checkpoint": str(CHECKPOINT_PATH),
    "input_checkpoint_size_gb": checkpoint_size_gb,
}


ENVIRONMENT_AUDIT_PATH = (
    FIGURE1D_DIR
    / "figure1d_environment_audit.json"
)


with open(
    ENVIRONMENT_AUDIT_PATH,
    "w",
) as file:
    json.dump(
        environment_audit,
        file,
        indent=2,
        default=str,
    )


print(
    "\nEnvironment audit saved to:\n"
    f"{ENVIRONMENT_AUDIT_PATH}"
)


# -----------------------------------------------------------------------------
# 12. Final validation report
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("CELL 4 VALIDATION REPORT")
print("=" * 80)

print(f"Rscript available: {rscript_path is not None}")
print(f"Missing R packages: {missing_r_packages}")
print(f"Available RAM: {available_ram_gb:.2f} GB")
print(f"Free disk space: {disk_free_gb:.2f} GB")
print(f"Input checkpoint exists: {CHECKPOINT_PATH.exists()}")

environment_ready = (
    rscript_path is not None
    and len(missing_r_packages) == 0
    and CHECKPOINT_PATH.exists()
    and available_ram_gb >= 40
    and disk_free_gb >= 15
)

print(f"Environment ready for full integration: {environment_ready}")

print("=" * 80)
print("CELL 4 COMPLETE")
print("=" * 80)

R EXECUTABLE CHECK
R path:       /usr/bin/R
Rscript path: /usr/bin/Rscript

Installed R version:
Rscript (R) version 4.5.0 (2025-04-11)

R PACKAGE CHECK
package	installed	version
Seurat	TRUE	5.5.1
Matrix	TRUE	1.7.3
zellkonverter	FALSE	NA
SingleCellExperiment	TRUE	1.32.0
glmGamPoi	TRUE	1.22.0
future	TRUE	1.75.0

Parsed package status:
  Seurat                   installed=True  version=5.5.1
  Matrix                   installed=True  version=1.7.3
  zellkonverter            installed=False version=None
  SingleCellExperiment     installed=True  version=1.32.0
  glmGamPoi                installed=True  version=1.22.0
  future                   installed=True  version=1.75.0

MEMORY CHECK
Total RAM:     62.80 GB
Used RAM:      5.97 GB
Available RAM: 56.83 GB
Memory usage:  9.5%

DISK CHECK
Disk total: 48.98 GB
Disk used:  16.23 GB
Disk free:  30.64 GB

INPUT CHECKPOINT
Checkpoint path: /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/results/figure1d_python/G

In [10]:
# =============================================================================
# CHECK ALL R LIBRARY LOCATIONS AND FIND INSTALLED PACKAGES
# =============================================================================
#
# Purpose
# -------
# 1. Show every R library path visible to the current R installation.
# 2. Search those paths for the required packages.
# 3. Search common system and Conda locations for additional R installations.
# 4. Report the exact package directory and package version.
#
# This cell does not install or modify anything.
# =============================================================================

import os
import shutil
import subprocess
from pathlib import Path

import pandas as pd


# -----------------------------------------------------------------------------
# 1. Locate all visible R and Rscript executables
# -----------------------------------------------------------------------------

print("=" * 80)
print("R EXECUTABLES ON PATH")
print("=" * 80)

for executable in ["R", "Rscript"]:
    path = shutil.which(executable)
    print(f"{executable}: {path}")


# Also search common locations for additional Rscript executables.
common_rscript_locations = [
    "/usr/bin/Rscript",
    "/usr/local/bin/Rscript",
    "/opt/conda/bin/Rscript",
    str(Path.home() / "miniconda3/bin/Rscript"),
    str(Path.home() / "anaconda3/bin/Rscript"),
    str(Path.home() / "mambaforge/bin/Rscript"),
    str(Path.home() / "miniforge3/bin/Rscript"),
]

found_rscripts = []

for candidate in common_rscript_locations:
    candidate_path = Path(candidate)

    if candidate_path.exists() and os.access(candidate_path, os.X_OK):
        found_rscripts.append(str(candidate_path.resolve()))

# Add the PATH-resolved Rscript if present.
path_rscript = shutil.which("Rscript")

if path_rscript is not None:
    found_rscripts.append(str(Path(path_rscript).resolve()))

found_rscripts = sorted(set(found_rscripts))

print("\nRscript executables found:")
for path in found_rscripts:
    print(f"  - {path}")


# -----------------------------------------------------------------------------
# 2. Ask each R installation for its library paths and package locations
# -----------------------------------------------------------------------------

required_packages = [
    "Seurat",
    "Matrix",
    "zellkonverter",
    "SingleCellExperiment",
    "glmGamPoi",
    "future",
]

all_results = []


r_code = r'''
packages <- c(
    "Seurat",
    "Matrix",
    "zellkonverter",
    "SingleCellExperiment",
    "glmGamPoi",
    "future"
)

cat("R_VERSION\t", R.version.string, "\n", sep = "")
cat("R_HOME\t", R.home(), "\n", sep = "")

for (lib in .libPaths()) {
    cat("LIBPATH\t", lib, "\n", sep = "")
}

for (pkg in packages) {

    installed <- requireNamespace(
        pkg,
        quietly = TRUE
    )

    if (installed) {

        pkg_path <- find.package(
            pkg,
            quiet = TRUE
        )

        pkg_version <- as.character(
            packageVersion(pkg)
        )

        cat(
            "PACKAGE\t",
            pkg, "\t",
            "TRUE", "\t",
            pkg_version, "\t",
            pkg_path, "\n",
            sep = ""
        )

    } else {

        cat(
            "PACKAGE\t",
            pkg, "\t",
            "FALSE", "\t",
            "NA", "\t",
            "NA", "\n",
            sep = ""
        )
    }
}
'''


for rscript in found_rscripts:

    print("\n" + "=" * 80)
    print(f"CHECKING R INSTALLATION: {rscript}")
    print("=" * 80)

    result = subprocess.run(
        [rscript, "-e", r_code],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        print("This R installation could not be queried.")
        print(result.stderr)
        continue

    print(result.stdout)

    current_r_version = None
    current_r_home = None
    current_libpaths = []

    for line in result.stdout.splitlines():

        fields = line.split("\t")

        if not fields:
            continue

        record_type = fields[0]

        if record_type == "R_VERSION" and len(fields) >= 2:
            current_r_version = fields[1]

        elif record_type == "R_HOME" and len(fields) >= 2:
            current_r_home = fields[1]

        elif record_type == "LIBPATH" and len(fields) >= 2:
            current_libpaths.append(fields[1])

        elif record_type == "PACKAGE" and len(fields) >= 5:

            all_results.append(
                {
                    "rscript": rscript,
                    "r_version": current_r_version,
                    "r_home": current_r_home,
                    "package": fields[1],
                    "installed": fields[2] == "TRUE",
                    "version": None if fields[3] == "NA" else fields[3],
                    "package_path": None if fields[4] == "NA" else fields[4],
                    "library_paths": "; ".join(current_libpaths),
                }
            )


# -----------------------------------------------------------------------------
# 3. Display a consolidated package-location table
# -----------------------------------------------------------------------------

results_df = pd.DataFrame(all_results)

print("\n" + "=" * 80)
print("CONSOLIDATED PACKAGE LOCATIONS")
print("=" * 80)

if len(results_df) == 0:
    print("No R installations could be queried.")
else:
    display(
        results_df[
            [
                "package",
                "installed",
                "version",
                "package_path",
                "rscript",
                "r_version",
            ]
        ].sort_values(
            ["package", "installed"],
            ascending=[True, False],
        )
    )


# -----------------------------------------------------------------------------
# 4. Search the filesystem for package directories outside visible .libPaths()
#
# This can find packages installed under another R or Conda environment.
# We restrict the search to common locations so it does not scan the entire disk.
# -----------------------------------------------------------------------------

search_roots = [
    Path("/usr/lib/R"),
    Path("/usr/local/lib/R"),
    Path("/usr/local/lib"),
    Path("/opt/conda"),
    Path.home() / "R",
    Path.home() / "miniconda3",
    Path.home() / "anaconda3",
    Path.home() / "mambaforge",
    Path.home() / "miniforge3",
]

filesystem_matches = []

print("\n" + "=" * 80)
print("FILESYSTEM SEARCH FOR PACKAGE DIRECTORIES")
print("=" * 80)

for root in search_roots:

    if not root.exists():
        continue

    print(f"Searching: {root}")

    for package in required_packages:

        try:
            for match in root.rglob(package):

                if not match.is_dir():
                    continue

                # A real installed R package directory normally contains
                # DESCRIPTION and NAMESPACE files.
                if (
                    (match / "DESCRIPTION").exists()
                    and (match / "NAMESPACE").exists()
                ):
                    filesystem_matches.append(
                        {
                            "package": package,
                            "path": str(match),
                            "search_root": str(root),
                        }
                    )

        except PermissionError:
            print(f"Permission denied while searching {root}")


filesystem_df = pd.DataFrame(filesystem_matches).drop_duplicates()

if len(filesystem_df) == 0:
    print("\nNo additional installed package directories were found.")
else:
    print("\nAdditional package directories found:")
    display(
        filesystem_df.sort_values(
            ["package", "path"]
        )
    )


# -----------------------------------------------------------------------------
# 5. Print environment variables that affect R library selection
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("R-RELATED ENVIRONMENT VARIABLES")
print("=" * 80)

for variable in [
    "R_HOME",
    "R_LIBS",
    "R_LIBS_USER",
    "R_LIBS_SITE",
    "CONDA_PREFIX",
    "PATH",
]:
    print(f"{variable}={os.environ.get(variable)}")


# -----------------------------------------------------------------------------
# 6. Final summary
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("PACKAGE LOCATION SUMMARY")
print("=" * 80)

if len(results_df) > 0:

    for package in required_packages:

        package_rows = results_df[
            (results_df["package"] == package)
            & (results_df["installed"])
        ]

        if len(package_rows) == 0:
            print(f"{package}: not visible to any queried R installation")

        else:
            print(f"{package}:")

            for _, row in package_rows.iterrows():
                print(
                    f"  version {row['version']} at {row['package_path']}"
                    f"\n  used by {row['rscript']}"
                )

print("=" * 80)
print("SEARCH COMPLETE")
print("=" * 80)

R EXECUTABLES ON PATH
R: /usr/bin/R
Rscript: /usr/bin/Rscript

Rscript executables found:
  - /usr/bin/Rscript

CHECKING R INSTALLATION: /usr/bin/Rscript
R_VERSION	R version 4.5.0 (2025-04-11)
R_HOME	/usr/lib/R
LIBPATH	/home/amithsiri/R/x86_64-pc-linux-gnu-library/4.5
LIBPATH	/usr/local/lib/R/site-library
LIBPATH	/usr/lib/R/site-library
LIBPATH	/usr/lib/R/library
PACKAGE	Seurat	TRUE	5.5.1	/home/amithsiri/R/x86_64-pc-linux-gnu-library/4.5/Seurat
PACKAGE	Matrix	TRUE	1.7.3	/usr/lib/R/library/Matrix
PACKAGE	zellkonverter	FALSE	NA	NA
PACKAGE	SingleCellExperiment	TRUE	1.32.0	/home/amithsiri/R/x86_64-pc-linux-gnu-library/4.5/SingleCellExperiment
PACKAGE	glmGamPoi	TRUE	1.22.0	/home/amithsiri/R/x86_64-pc-linux-gnu-library/4.5/glmGamPoi
PACKAGE	future	TRUE	1.75.0	/home/amithsiri/R/x86_64-pc-linux-gnu-library/4.5/future


CONSOLIDATED PACKAGE LOCATIONS


,package,installed,version,package_path,rscript,r_version
1,Matrix,True,1.7.3,/usr/lib/R/library/Matrix,/usr/bin/Rscript,R version 4.5.0 (2025-04-11)
0,Seurat,True,5.5.1,/home/amithsiri/R/x86_64-pc-linux-gnu-library/...,/usr/bin/Rscript,R version 4.5.0 (2025-04-11)
3,SingleCellExperiment,True,1.32.0,/home/amithsiri/R/x86_64-pc-linux-gnu-library/...,/usr/bin/Rscript,R version 4.5.0 (2025-04-11)
5,future,True,1.75.0,/home/amithsiri/R/x86_64-pc-linux-gnu-library/...,/usr/bin/Rscript,R version 4.5.0 (2025-04-11)
4,glmGamPoi,True,1.22.0,/home/amithsiri/R/x86_64-pc-linux-gnu-library/...,/usr/bin/Rscript,R version 4.5.0 (2025-04-11)
2,zellkonverter,False,NaN,NaN,/usr/bin/Rscript,R version 4.5.0 (2025-04-11)



FILESYSTEM SEARCH FOR PACKAGE DIRECTORIES
Searching: /usr/lib/R
Searching: /usr/local/lib/R
Searching: /usr/local/lib
Searching: /home/amithsiri/R

Additional package directories found:


,package,path,search_root
0,Matrix,/usr/lib/R/library/Matrix,/usr/lib/R
1,Seurat,/home/amithsiri/R/library/Seurat,/home/amithsiri/R
2,Seurat,/home/amithsiri/R/x86_64-pc-linux-gnu-library/...,/home/amithsiri/R
3,SingleCellExperiment,/home/amithsiri/R/x86_64-pc-linux-gnu-library/...,/home/amithsiri/R
5,future,/home/amithsiri/R/library/future,/home/amithsiri/R
6,future,/home/amithsiri/R/x86_64-pc-linux-gnu-library/...,/home/amithsiri/R
4,glmGamPoi,/home/amithsiri/R/x86_64-pc-linux-gnu-library/...,/home/amithsiri/R



R-RELATED ENVIRONMENT VARIABLES
R_HOME=None
R_LIBS=None
R_LIBS_USER=None
R_LIBS_SITE=None
CONDA_PREFIX=None
PATH=/home/amithsiri/.venvs/jupyter/bin:/home/amithsiri/.vscode-server/cli/servers/Stable-1b6a188127eeaf9194f945eb6eb89a657e93c54c/server/bin/remote-cli:/usr/local/bin:/usr/bin:/bin:/usr/local/games:/usr/games

PACKAGE LOCATION SUMMARY
Seurat:
  version 5.5.1 at /home/amithsiri/R/x86_64-pc-linux-gnu-library/4.5/Seurat
  used by /usr/bin/Rscript
Matrix:
  version 1.7.3 at /usr/lib/R/library/Matrix
  used by /usr/bin/Rscript
zellkonverter: not visible to any queried R installation
SingleCellExperiment:
  version 1.32.0 at /home/amithsiri/R/x86_64-pc-linux-gnu-library/4.5/SingleCellExperiment
  used by /usr/bin/Rscript
glmGamPoi:
  version 1.22.0 at /home/amithsiri/R/x86_64-pc-linux-gnu-library/4.5/glmGamPoi
  used by /usr/bin/Rscript
future:
  version 1.75.0 at /home/amithsiri/R/x86_64-pc-linux-gnu-library/4.5/future
  used by /usr/bin/Rscript
SEARCH COMPLETE


In [11]:
# =============================================================================
# CELL 6 — EXPORT THE LOADED ANNDATA OBJECT FOR SEURAT
# =============================================================================
#
# Purpose
# -------
# Export the already loaded AnnData object into files that R/Seurat can read:
#
#   counts.mtx
#   genes.tsv
#   barcodes.tsv
#   metadata.csv
#
# This avoids needing zellkonverter.
#
# The count matrix remains sparse.
# =============================================================================


from pathlib import Path

import pandas as pd
import scipy.io
import scipy.sparse as sp


# -----------------------------------------------------------------------------
# 1. Define the export directory
# -----------------------------------------------------------------------------

SEURAT_INPUT_DIR = Path(
    "/home/amithsiri/projects/"
    "Summer-Project-2026/"
    "Summer-Project-2026/"
    "Age_trajectory/"
    "results/"
    "figure1d_python/"
    "seurat_input"
)

SEURAT_INPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# -----------------------------------------------------------------------------
# 2. Define output file paths
# -----------------------------------------------------------------------------

MTX_PATH = SEURAT_INPUT_DIR / "counts.mtx"
GENES_PATH = SEURAT_INPUT_DIR / "genes.tsv"
BARCODES_PATH = SEURAT_INPUT_DIR / "barcodes.tsv"
METADATA_PATH = SEURAT_INPUT_DIR / "metadata.csv"


print("=" * 80)
print("SEURAT EXPORT PATHS")
print("=" * 80)

print(f"Matrix:    {MTX_PATH}")
print(f"Genes:     {GENES_PATH}")
print(f"Barcodes:  {BARCODES_PATH}")
print(f"Metadata:  {METADATA_PATH}")


# -----------------------------------------------------------------------------
# 3. Select the preserved count matrix
#
# AnnData stores:
#
#   rows    = cells
#   columns = genes
#
# Seurat expects:
#
#   rows    = genes
#   columns = cells
#
# Therefore, the matrix must be transposed before export.
# -----------------------------------------------------------------------------

if "counts" not in adata.layers:
    raise KeyError(
        "adata.layers['counts'] was not found."
    )


counts = adata.layers["counts"]


if not sp.issparse(counts):
    raise TypeError(
        "The count layer is not sparse."
    )


counts = counts.tocsr()


print("\nAnnData count matrix:")
print(f"Shape: {counts.shape}")
print(f"Sparse type: {type(counts)}")
print(f"Nonzero values: {counts.nnz:,}")


# Transpose to genes × cells for Seurat.
counts_for_seurat = counts.T.tocoo()


print("\nSeurat-oriented count matrix:")
print(f"Shape: {counts_for_seurat.shape}")
print(f"Sparse type: {type(counts_for_seurat)}")


# -----------------------------------------------------------------------------
# 4. Export the sparse Matrix Market file
#
# This may take several minutes because the matrix contains approximately
# 126 million nonzero entries.
# -----------------------------------------------------------------------------

if MTX_PATH.exists():
    print(
        "\ncounts.mtx already exists and will not be overwritten."
    )
else:
    print("\nWriting sparse Matrix Market file...")

    scipy.io.mmwrite(
        MTX_PATH,
        counts_for_seurat,
        field="real",
    )

    print("Matrix export complete.")


# -----------------------------------------------------------------------------
# 5. Export gene names
# -----------------------------------------------------------------------------

gene_names = pd.Series(
    adata.var_names.astype(str),
    name="gene",
)


if not gene_names.is_unique:
    raise ValueError(
        "Gene names are not unique."
    )


gene_names.to_csv(
    GENES_PATH,
    sep="\t",
    index=False,
    header=False,
)


print(f"\nGenes exported: {len(gene_names):,}")


# -----------------------------------------------------------------------------
# 6. Export cell barcodes
# -----------------------------------------------------------------------------

cell_barcodes = pd.Series(
    adata.obs_names.astype(str),
    name="barcode",
)


if not cell_barcodes.is_unique:
    raise ValueError(
        "Cell barcodes are not unique."
    )


cell_barcodes.to_csv(
    BARCODES_PATH,
    sep="\t",
    index=False,
    header=False,
)


print(f"Barcodes exported: {len(cell_barcodes):,}")


# -----------------------------------------------------------------------------
# 7. Export cell metadata
#
# The metadata row names must match the cell barcodes exactly.
# We include the barcode as the first explicit column for reliable matching in R.
# -----------------------------------------------------------------------------

metadata_export = adata.obs.copy()

metadata_export.insert(
    0,
    "cell_barcode",
    adata.obs_names.astype(str),
)


metadata_export.to_csv(
    METADATA_PATH,
    index=False,
)


print(f"Metadata rows exported: {len(metadata_export):,}")
print(f"Metadata columns exported: {metadata_export.shape[1]:,}")


# -----------------------------------------------------------------------------
# 8. Validate dimensions
# -----------------------------------------------------------------------------

expected_matrix_shape = (
    adata.n_vars,
    adata.n_obs,
)


if counts_for_seurat.shape != expected_matrix_shape:
    raise ValueError(
        f"Export matrix shape is {counts_for_seurat.shape}, "
        f"expected {expected_matrix_shape}."
    )


if len(gene_names) != counts_for_seurat.shape[0]:
    raise ValueError(
        "Gene count does not match matrix rows."
    )


if len(cell_barcodes) != counts_for_seurat.shape[1]:
    raise ValueError(
        "Barcode count does not match matrix columns."
    )


if len(metadata_export) != counts_for_seurat.shape[1]:
    raise ValueError(
        "Metadata rows do not match matrix columns."
    )


# -----------------------------------------------------------------------------
# 9. Report file sizes
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("EXPORTED FILE SIZES")
print("=" * 80)


for path in [
    MTX_PATH,
    GENES_PATH,
    BARCODES_PATH,
    METADATA_PATH,
]:
    size_gb = path.stat().st_size / (1024**3)

    print(
        f"{path.name:<15} "
        f"{size_gb:.3f} GB"
    )


# -----------------------------------------------------------------------------
# 10. Store paths for later cells
# -----------------------------------------------------------------------------

SEURAT_EXPORT_PATHS = {
    "matrix": str(MTX_PATH),
    "genes": str(GENES_PATH),
    "barcodes": str(BARCODES_PATH),
    "metadata": str(METADATA_PATH),
}


# -----------------------------------------------------------------------------
# 11. Final validation report
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("CELL 6 VALIDATION REPORT")
print("=" * 80)

print(
    "Exported matrix dimensions: "
    f"{counts_for_seurat.shape[0]:,} genes × "
    f"{counts_for_seurat.shape[1]:,} cells"
)

print(
    "Expected matrix dimensions: "
    f"{adata.n_vars:,} genes × "
    f"{adata.n_obs:,} cells"
)

print(f"Matrix file exists: {MTX_PATH.exists()}")
print(f"Gene file exists: {GENES_PATH.exists()}")
print(f"Barcode file exists: {BARCODES_PATH.exists()}")
print(f"Metadata file exists: {METADATA_PATH.exists()}")

print("=" * 80)
print("CELL 6 COMPLETE")
print("=" * 80)

SEURAT EXPORT PATHS
Matrix:    /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/results/figure1d_python/seurat_input/counts.mtx
Genes:     /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/results/figure1d_python/seurat_input/genes.tsv
Barcodes:  /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/results/figure1d_python/seurat_input/barcodes.tsv
Metadata:  /home/amithsiri/projects/Summer-Project-2026/Summer-Project-2026/Age_trajectory/results/figure1d_python/seurat_input/metadata.csv

AnnData count matrix:
Shape: (70391, 19689)
Sparse type: <class 'scipy.sparse._csr.csr_matrix'>
Nonzero values: 126,603,213

Seurat-oriented count matrix:
Shape: (19689, 70391)
Sparse type: <class 'scipy.sparse._coo.coo_matrix'>

Writing sparse Matrix Market file...
Matrix export complete.

Genes exported: 19,689
Barcodes exported: 70,391
Metadata rows exported: 70,391
Metadata columns exported: 17

EXPORTED FILE SIZ